In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py

from tqdm import tqdm

from glob import glob

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
def GetPlottingDirectory(plotFileName, plotType):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz")
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

def SaveFigure(fig,plotType, fileName):
    plotFileName = f"{fileName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType)
    print(f"Saving figure to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
path=os.path.join(dir2,'Code/CodeFiles/Functions')
sys.path.append(path)

import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
#############################################
#LOADING DATA

In [ ]:
#READING BACK IN SUBSETTED TRACKED PARCEL DATA
trackedArrays,LevelsDictionary = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,
                                                         Results_InputOutput_Class)

In [ ]:
def job_filter(arr, start_job,end_job):
        return arr[(arr[:,1]>=start_job)&(arr[:,1]<end_job)]
def FilterTrackedArrays(trackedArrays, start_job, end_job):
    filtered = {}
    for category, subDict in trackedArrays.items():
        filtered[category] = {}
        for key, out_arr in subDict.items():
            if out_arr is None or len(out_arr) == 0:
                filtered[category][key] = out_arr
            else:
                filtered[category][key] = job_filter(out_arr, start_job, end_job)
    return filtered

trackedArraysFiltered = FilterTrackedArrays(trackedArrays, start_job, end_job)
del trackedArrays

In [ ]:
#GET SBF LOCATIONS
def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence
    
def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs

def Get_SBF_xmaxs_FilePath(ModelData, DataManager):
    fileName = (
        f"SBF_xmaxs_{ModelData.res}_{ModelData.t_res}_"
        f"{ModelData.Ntime}nt.pkl"
    )
    return os.path.join(DataManager.outputDataDirectory, fileName)

def LoadOrRun_find_SBF_xmaxs(ModelData, DataManager, overwrite=False):
    filePath = Get_SBF_xmaxs_FilePath(ModelData, DataManager)
    
    if os.path.exists(filePath) and not overwrite:
        print(f"reading from {filePath}")
        with open(filePath, "rb") as f:
            xmaxs = pickle.load(f)
        return xmaxs

    xmaxs = find_SBF_xmaxs()

    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump(xmaxs, f)
    print(f"saved to {filePath}")
    return xmaxs

xmaxs = LoadOrRun_find_SBF_xmaxs(ModelData, DataManager)

In [ ]:
def GetThetaMean(t):
    xmaxs_t=xmaxs[t]
    timeString=ModelData.timeStrings[t]
    theta_v = CallVariable(ModelData, DataManager, timeString, variableName='theta_v')[0]
    theta_e = CallVariable(ModelData, DataManager, timeString, variableName='theta_e')[0]
    if np.isnan(xmaxs_t):
        return np.nan,np.nan
    theta_v_mean=np.mean(theta_v[:,xmaxs_t+10:])
    theta_e_mean=np.mean(theta_e[:,xmaxs_t+10:])
    return theta_v_mean,theta_e_mean

def CalculateMeansForPerturbations():
    theta_v_mean,theta_e_mean=[],[]
    for t in tqdm(range(ModelData.Ntime)):
        out = GetThetaMean(t)
        theta_v_mean.append(out[0]);theta_e_mean.append(out[1])
    return theta_v_mean,theta_e_mean

def Get_CalculateMeansForPerturbations_FilePath(ModelData, DataManager):
    fileName = (
        f"MeansForPerturbations_{ModelData.res}_{ModelData.t_res}_"
        f"{ModelData.Ntime}nt.pkl"
    )
    return os.path.join(DataManager.outputDataDirectory, fileName)

def CalculateMeansForPerturbations_LoadOrRun(overwrite=False):
    filePath = Get_CalculateMeansForPerturbations_FilePath(ModelData, DataManager)

    if os.path.exists(filePath) and not overwrite:
        print(f"reading from {filePath}")
        with open(filePath, "rb") as f:
            outputDictionary = pickle.load(f)

        return outputDictionary["theta_v_mean"], outputDictionary["theta_e_mean"]

    theta_v_mean, theta_e_mean = CalculateMeansForPerturbations()

    outputDictionary = {
        "theta_v_mean": theta_v_mean,
        "theta_e_mean": theta_e_mean,
    }

    os.makedirs(os.path.dirname(filePath), exist_ok=True)

    with open(filePath, "wb") as f:
        pickle.dump(outputDictionary, f)

    print(f"saved to {filePath}")

    return theta_v_mean, theta_e_mean

# time=np.arange(ModelData.Ntime)*ModelData.dt/3600+6;
# plt.plot(time,theta_v_mean)
# plt.xlabel('time (hrs)');plt.ylabel(r'$\theta_v\ (K)$');

# time=np.arange(ModelData.Ntime)*ModelData.dt/3600+6;
# plt.plot(time,theta_e_mean)
# plt.xlabel('time (hrs)');plt.ylabel(r'$\theta_e\ (K)$');

[theta_v_mean,theta_e_mean] = CalculateMeansForPerturbations_LoadOrRun()

In [ ]:
def Get_LagrangianArrays(t, dataType="VARS", dataName="VARS", varNames=["W"]):
    res = ModelData.res
    t_res = ModelData.t_res
    Nz_str = ModelData.Nz_str
    inputDirectory = os.path.join(DataManager.inputDirectory,
                                  "..","LagrangianArrays",
                                  f"{res}_{t_res}_{Nz_str}nz", dataType)
    timeString = ModelData.timeStrings[t]

    FileName = os.path.join(inputDirectory, f"{dataName}_{res}_{t_res}_{Nz_str}nz_{timeString}.h5")

    dataDictionary = {}
    with h5py.File(FileName, 'r') as f:
        # print("Keys in file:", list(f.keys()))
        for key in varNames:
            dataDictionary[key] = f[key][:]
            # print(f"{key}: shape = {dataDictionary[key].shape}, dtype = {dataDictionary[key].dtype}")
    return dataDictionary

In [ ]:
#############################################
#HISTOGRAM RUNNING FUNCTIONS

In [ ]:
#numerical info
zh = ModelData.zh
yh = ModelData.yh-ModelData.yh[0]
xh = ModelData.xh-ModelData.xh[0]
# kms=np.argmax(ModelData.xh-ModelData.xh[0] >= 1)

In [ ]:
def CollectData(trackedArray):
    #getting parcel index and time
    ps = trackedArray[:,0]
    ts = trackedArray[:,1]

    # sort by time
    sort_idx = np.argsort(ts)
    ts_sorted = ts[sort_idx]
    ps_sorted = ps[sort_idx]

    #initializing lists
    T_List = []; Z_List = []; Y_List = []; X_List = []
    Xdiff_List = []
    QV_List,THETA_v_List,THETA_e_List = [],[],[]
    THETA_v_prime_List,THETA_e_prime_List = [],[]

    #time cache (to avoid redundant looping)
    previous_t = None
    
    #running through each parcel
    for t, p in tqdm(
        zip(ts_sorted, ps_sorted),
        total=len(ts_sorted),
        desc="Processing timesteps"):

        #X and VARS loading
        if t != previous_t:
            previous_t = t
            timeString = ModelData.timeStrings[t]

            Z_t = CallLagrangianArray(ModelData, DataManager, timeString, 'Z')
            Y_t = CallLagrangianArray(ModelData, DataManager, timeString, 'Y')
            X_t = CallLagrangianArray(ModelData, DataManager, timeString, 'X')
            xmaxs_t = xh[xmaxs[t]]
            
            VARS=Get_LagrangianArrays(t,varNames=["QV","THETA_v",'THETA_e'])
            QV_t = VARS["QV"]; THETA_v_t = VARS["THETA_v"]; THETA_e_t = VARS["THETA_e"]

        #DISTANCE METRICS
        Z_tp = Z_t[p] #getting z-grid number
        Y_tp = Y_t[p] #getting y-grid number
        X_tp = X_t[p] #getting x-grid number
        #converting to km
        Z_tp = zh[Z_tp]
        Y_tp = yh[Y_tp]
        X_tp = xh[X_tp]
    
        #getting index distance from sea-breeze
        Xdiff = X_tp - xmaxs_t
    
        #appending results to list
        T_List.append(t)
        Z_List.append(Z_tp)
        Y_List.append(Y_tp)
        X_List.append(X_tp)
        Xdiff_List.append(Xdiff)

        #VARIABLES
        QVParcel_t = QV_t[p]
        THETA_vParcel_t = THETA_v_t[p]; THETA_eParcel_t = THETA_e_t[p]
        QV_List.append(QVParcel_t); THETA_v_List.append(THETA_vParcel_t); THETA_e_List.append(THETA_eParcel_t)

        THETA_v_prime_Parcel_t = THETA_vParcel_t - theta_v_mean[t]; THETA_e_prime_Parcel_t = THETA_eParcel_t - theta_e_mean[t]
        THETA_v_prime_List.append(THETA_v_prime_Parcel_t);THETA_e_prime_List.append(THETA_e_prime_Parcel_t)
    
        
    return (T_List,Z_List,Y_List,X_List,Xdiff_List, 
            QV_List,THETA_v_List,THETA_e_List,
            THETA_v_prime_List,THETA_e_prime_List)

In [ ]:
def RunAllParcelTypes(trackedArraysFiltered):
    results = {}
    
    # for outer_key, inner_dict in trackedArraysFiltered.items():          # e.g. "CL"
    for outer_key, inner_dict in tqdm(trackedArraysFiltered.items()):
        results[outer_key] = {}
        for inner_key, trackedArray in inner_dict.items():       # e.g. "DEEP"
            print(f"\nRunning CollectData for {outer_key} - {inner_key}")
    
            if trackedArray is None or len(trackedArray) == 0:
                print(f"  Skipping {outer_key}-{inner_key}: empty array")
                continue
    
            [T_List,Z_List,Y_List,X_List,Xdiff_List, 
            QV_List,THETA_v_List,THETA_e_List,
            THETA_v_prime_List,THETA_e_prime_List] = CollectData(trackedArray)
    
            # store results in nested dict
            results[outer_key][inner_key] = {
                "T_List": T_List,
                "Z_List": Z_List,
                "Y_List": Y_List,
                "X_List": X_List,
                "Xdiff_List": Xdiff_List,
                
                "QV_List": QV_List,
                "THETA_v_List": THETA_v_List,
                "THETA_e_List": THETA_e_List,
                
                "THETA_v_prime_List": THETA_v_prime_List,
                "THETA_e_prime_List": THETA_e_prime_List,
            }
    return results

In [ ]:
def LoadorRun():
    """
    Loads the tracked parcel results from a pickle file if it exists;
    otherwise runs RunAllParcelTypes() and saves the output.
    """
    fileName = f"Tracked_Histogram_Output_{ModelData.res}_{ModelData.t_res}_{ModelData.Nzh}nz_{SlurmJobArray.start_job}-{SlurmJobArray.end_job}.pkl"
    folderPath = os.path.join(DataManager.outputDirectory, f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz", "Tracked_Histograms")
    os.makedirs(folderPath, exist_ok=True)
    filePath = os.path.join(folderPath, fileName)

    if os.path.exists(filePath):
        # Load existing results
        with open(filePath, "rb") as f:
            results = pickle.load(f)
        print(f"Loaded results from {filePath}")
    else:
        # Run function and save new results
        print(f"No pickle file found, running RunAllParcelTypes()...")
        results = RunAllParcelTypes(trackedArraysFiltered)

        with open(filePath, "wb") as f:
            pickle.dump(results, f)
        print(f"Saved results to {filePath}")

    return results

In [ ]:
#############################################
#HISTOGRAM RUNNING
running = True #keep True when job array is running
# running = False

In [ ]:
if running:
    results = LoadorRun()

In [ ]:
#############################################
#HISTOGRAM RECOMBINING
recombining = False #keep False when job array is running
recombining = True 

In [ ]:
def OpenFile(ModelData,SlurmJobArray, start_job,end_job):
    fileName = f"Tracked_Histogram_Output_{ModelData.res}_{ModelData.t_res}_{ModelData.Nzh}nz_{start_job}-{end_job}.pkl"
    folderPath = os.path.join(DataManager.outputDirectory, f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz", "Tracked_Histograms")
    filePath = os.path.join(folderPath, fileName)

    with open(filePath, "rb") as f:
        results = pickle.load(f)
    return results

def InitializeCombinedResults(trackedArraysFiltered):
    results = {}

    for outer_key, inner_dict in trackedArraysFiltered.items():
        results[outer_key] = {}

        for inner_key in inner_dict.keys():
            results[outer_key][inner_key] = {
                "T_List": [], "Z_List": [], "Y_List": [], "X_List": [],
                "Xdiff_List": [],
                "QV_List": [], "THETA_v_List": [], "THETA_e_List": [],
                "THETA_v_prime_List": [], "THETA_e_prime_List": []
            }

    return results

def Recombine(ModelData):
    resultsCombined = InitializeCombinedResults(trackedArraysFiltered)
    
    for job_id in tqdm(range(1, num_jobs + 1)):
        start_job, end_job = SlurmJobArray._get_job_range(job_id)
    
        job_results = OpenFile(ModelData,SlurmJobArray, start_job,end_job)
    
        for outer_key, inner_dict in job_results.items():
            for inner_key, data_dict in inner_dict.items():
                for var_name, values in data_dict.items():
                    resultsCombined[outer_key][inner_key][var_name].extend(values)

        job_results.clear()
        del job_results
        if loop_elements[-1]==ModelData.Ntime-1: break
    return resultsCombined


def OpenOrLoad(ModelData):
    fileName = f"Tracked_Histogram_Output_{ModelData.res}_{ModelData.t_res}_{ModelData.Nzh}nz_combined.pkl"
    folderPath = os.path.join(DataManager.outputDirectory, f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz", "Tracked_Histograms")
    filePath = os.path.join(folderPath, fileName)
    
    # ---- try to load
    if os.path.exists(filePath):
        print(f"Loading combined results: {filePath}")
        with open(filePath, "rb") as f:
            results = pickle.load(f)
        return results

    # ---- otherwise recombine
    print("Combined file not found — recombining")
    results = Recombine(ModelData)

    # ---- SAVE combined output
    with open(filePath, "wb") as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved combined results: {filePath}")
    return results

In [ ]:
if recombining:
    results = OpenOrLoad(ModelData)

In [ ]:
#############################################
#HISTOGRAM CHECKING SIGNIFICANCE FUNCTION

In [ ]:
def RunStatisticalComparison(results, 
                             categories=["CL", "nonCL", "SBF", "ColdPool"],
                             depthTypes=["ALL", "SHALLOW", "DEEP"],
                             variableNames=["QV_List", "THETA_v_List"],
                             test="mannwhitney"):
    """
    Perform statistical comparison and generate one multi-panel subplot figure.
    Returns pvalues and the figure.
    """

    # All unique category pairs
    parcelTypePairs = [(a, b)
                       for i, a in enumerate(categories)
                       for b in categories[i+1:]]

    # Storage
    pvalues = {var: {} for var in variableNames}

    # Select test
    if test == "mannwhitney":
        test_func = lambda x, y: mannwhitneyu(x, y, alternative='two-sided')[1]
        title_name = "Mann–Whitney U Test"
    elif test == "brunnermunzel":
        test_func = lambda x, y: brunnermunzel(x, y, alternative='two-sided')[1]
        title_name = "Brunner–Munzel Test"
    elif test == "median":
        test_func = lambda x, y: median_test(x, y)[1]
        title_name = "Mood's Median Test"
    else:
        raise ValueError("test must be 'median', 'mannwhitney', or 'brunnermunzel'")
    #Example Interpretation
    #Mann-Whitney U Test
    # values from one group consistently rank higher than values from the other group far more often than would be expected if both groups came from the same underlying population.
    
    #Brunner Munzel test
    #The probability of X being greater than Y is not equal to the probability of Y being greater than X.

    #Moods Median Test:
    # The number of observations above the pooled median
    # and below the pooled median
    # is extremely different between the two groups.

    # --------------------------------------------------------
    # MAIN LOOP
    # --------------------------------------------------------
    for var in variableNames:
        for (p1, p2) in parcelTypePairs:
            for depth in depthTypes:
                try:
                    arr1 = results[p1][depth][var]
                    arr2 = results[p2][depth][var]
                    p = test_func(arr1, arr2)
                except Exception:
                    p = np.nan
                pvalues[var][(p1, p2, depth)] = p

    # --------------------------------------------------------
    # BUILD SUBPLOTS
    # --------------------------------------------------------
    nVars = len(variableNames)
    fig, axes = plt.subplots(nVars, 1, figsize=(6*nVars, 8), squeeze=False)
    axes = axes.flatten()

    for ax, var in zip(axes, variableNames):

        keys = list(pvalues[var].keys())
        labels = [f"{a} vs {b}\n({d})" for (a, b, d) in keys]
        pv = [pvalues[var][k] for k in keys]

        ax.plot(pv, "o-")
        ax.axhline(0.05, color="red", linestyle="--")

        ax.set_title(f"{var}")
        ax.set_xticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=60, ha="right")
        ax.set_ylim(0, 1)
        ax.set_ylabel("p-value")

    plt.suptitle(f"{title_name} for All Variables", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.93])

    return pvalues, fig
from scipy.stats import mannwhitneyu, brunnermunzel, median_test

In [ ]:
#############################################
#HISTOGRAM PLOTTING FUNCTIONS

In [ ]:
#PlotHistogram

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Helvetica", "Arial"]
plt.rcParams["axes.titlesize"] = 8
plt.rcParams["axes.titleweight"] = "normal"   # no bold
plt.rcParams["axes.labelsize"] = 7            # smaller axis labels
plt.rcParams["xtick.labelsize"] = 10           # smaller tick labels
plt.rcParams["ytick.labelsize"] = 10

def PlotHistogram(axis, dataList, xlabel, bins=50, orientation="vertical",
                  color='steelblue', title=None, 
                  plotKDE=True):

    data = np.array(dataList)

    # histogram
    counts, bin_edges, _ = axis.hist(
        data,
        bins=bins,
        color=color,
        edgecolor='black',
        alpha=0.7,
        orientation=orientation        # <-- THIS FIXES YOUR PROBLEM
    )

    if plotKDE == True:
        # KDE
        kde = gaussian_kde(data)
        x_vals = np.linspace(bin_edges[0], bin_edges[-1], 400)
        bin_width = bin_edges[1] - bin_edges[0]
        kde_scaled = kde(x_vals) * len(data) * bin_width

        # plot KDE depending on orientation
        if orientation == "horizontal":
            axis.plot(
                kde_scaled, x_vals,
                color='blue', linewidth=1.8, zorder=10,
                label="_ignore_snap_"
            )
            axis.set_xlabel("Count")
            axis.set_ylabel(xlabel)
        else:
            axis.plot(
                x_vals, kde_scaled,
                color='blue', linewidth=1.8, zorder=10,
                label="_ignore_snap_"
            )
            axis.set_xlabel(xlabel)
            axis.set_ylabel("Count")
    
        if title:
            axis.set_title(title, fontsize=10, pad=10)
    
        axis.grid(True, linestyle='--', alpha=0.4)


In [ ]:
def PlotDistancesFunction(parcel_type):
    # choose which outer key to plot
    ptype = parcel_type
    depth_types = ["ALL", "SHALLOW", "DEEP"]
    
    # set up figure (2 rows × 3 columns)
    fig = plt.figure(figsize=(12, 8))
    gs  = gridspec.GridSpec(2, len(depth_types), figure=fig,
                            wspace=0.3, hspace=0.35)
    
    # loop through depth types
    for j, depth in enumerate(depth_types):
        # first row: X_List
        ax_top = fig.add_subplot(gs[0, j])
        if ptype in results and depth in results[ptype]:
            data_x = results[ptype][depth]["X_List"]
            data_x_mean = np.mean(data_x)
            PlotHistogram(ax_top, data_x,
                          xlabel="X distance from left side (km)",
                          title=f"{ptype} – {depth}\n" 
                          + r"$\mu$ = %.2f km" % data_x_mean)
    
        # second row: Xdiff_List
        ax_bottom = fig.add_subplot(gs[1, j])
        if ptype in results and depth in results[ptype]:
            data_xdiff = results[ptype][depth]["Xdiff_List"]
            data_xdiff_mean = np.mean(data_xdiff)
            PlotHistogram(ax_bottom, data_xdiff,
                          xlabel="X distance from SBF (km)",
                          title=f"{ptype} – {depth}\n"
                                + r"$\mu$ = %.2f km" % data_xdiff_mean)
        else:
            continue
 
    fig.subplots_adjust(left=0.07, right=0.97,   
                        bottom=0.08, top=0.90,
                        wspace=0.35, hspace=0.35)
    return fig

def PlotVariablesFunction(parcel_type):
    # choose which outer key to plot
    ptype = parcel_type
    depth_types = ["ALL", "SHALLOW", "DEEP"]
    
    # set up figure (2 rows × 3 columns)
    fig = plt.figure(figsize=(12, 8))
    gs  = gridspec.GridSpec(2, len(depth_types), figure=fig,
                            wspace=0.3, hspace=0.35)
    
    # loop through depth types
    for j, depth in enumerate(depth_types):
        # first row: QV
        ax_top = fig.add_subplot(gs[0, j])
        if ptype in results and depth in results[ptype]:
            data_x = results[ptype][depth]["QV_List"]
            data_x = np.array(data_x)*1e3
            data_x_mean = np.mean(data_x)
            PlotHistogram(ax_top, data_x,
                          xlabel="qv (g/kg)",
                          title=f"{ptype} – {depth}\n"
                                + r"$\mu$ = %.2f g/kg" % data_x_mean)
    
        # second row: TH
        ax_bottom = fig.add_subplot(gs[1, j])
        if ptype in results and depth in results[ptype]:
            data_x = results[ptype][depth]["THETA_v_List"]
            data_x_mean = np.mean(data_x)
            PlotHistogram(ax_bottom, data_x,
                          xlabel="th_v (K)",
                          title=f"{ptype} – {depth}\n"
                                + r"$\mu$ = %.2f K" % data_x_mean)
        else:
            continue
 
    fig.subplots_adjust(left=0.07, right=0.97,   
                        bottom=0.08, top=0.90,
                        wspace=0.35, hspace=0.35)
    return fig

# #############################################
# #SINGLE HISTOGRAM PLOTTING
# plotting = False #keep false when running job array
# # plotting = True

# if plotting:
#     parcel_types = ["CL", "nonCL", "SBF", "ColdPool"]
#     for parcel_type in parcel_types:
#         fig = PlotDistancesFunction(parcel_type)
    
#         #saving
#         fileName=f"Tracked_Histograms_Distances_{parcel_type}" 
#         SaveFigure(fig,plotType="Project_Algorithms/Tracking_Algorithms/Tracked_Histograms",fileName=fileName)

# if plotting:
#     parcel_types = ["CL", "nonCL", "SBF","ColdPool"]
#     for parcel_type in parcel_types:
#         fig = PlotVariablesFunction(parcel_type)
    
#         #saving
#         fileName=f"Tracked_Histograms_Variables_{parcel_type}" 
#         SaveFigure(fig,plotType="Project_Algorithms/Tracking_Algorithms/Tracked_Histograms",fileName=fileName)

In [ ]:
def PlotAllHistograms_Heights(parcel_types, results):
    """
    Creates a single figure with 4 parcel-type blocks:
      [CL, nonCL]
      [SBF, ColdPool]
    Each block contains a 2×3 grid of subplots (ALL, SHALLOW, DEEP).
    """

    depth_types = ["ALL", "SHALLOW", "DEEP"]

    # 2×2 outer grid for parcel-type groups
    fig = plt.figure(figsize=(14, 8))
    outer_gs = gridspec.GridSpec(2, 2, figure=fig, wspace=0.2, hspace=0.45)

    # map parcel types into positions
    parcel_map = {
        (0, 0): "CL",
        (0, 1): "nonCL",
        (1, 0): "SBF",
        (1, 1): "ColdPool"
    }

    # loop through outer 2×2 positions
    for (r, c), ptype in parcel_map.items():
        inner_gs = gridspec.GridSpecFromSubplotSpec(
            1, len(depth_types), subplot_spec=outer_gs[r, c],
            wspace=0.4, hspace=0.6  # <-- more breathing room between rows
        )

        # loop through 2×3 subplots inside each parcel-type block
        for j, depth in enumerate(depth_types):
            # --- TOP ROW: Z_List ---
            ax_top = fig.add_subplot(inner_gs[0, j])
            if ptype in results and depth in results[ptype]:
                data_z = np.array(results[ptype][depth]["Z_List"])*1000
                data_z_mean = np.mean(data_z)
                PlotHistogram(
                    ax_top, data_z,
                    orientation='horizontal',
                    xlabel="Z distance (m)",
                    title=f"{ptype} – {depth}\n"
                          + r"$\mu$ = %.2f m" % data_z_mean,
                     plotKDE=True
                )

                ax_top.set_ylim(bottom=0, top=1000)

    # Adjust overall layout to prevent overlap
    fig.subplots_adjust(left=0.06, right=0.97, bottom=0.06, top=0.94)

    AddCrossLines(fig, outer_gs)
    return fig

def PlotAllHistograms_Distances(parcel_types, results):
    """
    Creates a single figure with 4 parcel-type blocks:
      [CL, nonCL]
      [SBF, ColdPool]
    Each block contains a 2×3 grid of subplots (ALL, SHALLOW, DEEP).
    """

    depth_types = ["ALL", "SHALLOW", "DEEP"]

    # 2×2 outer grid for parcel-type groups
    fig = plt.figure(figsize=(14, 10))
    outer_gs = gridspec.GridSpec(2, 2, figure=fig, wspace=0.2, hspace=0.45)

    # map parcel types into positions
    parcel_map = {
        (0, 0): parcel_types[0],
        (0, 1): parcel_types[1],
        (1, 0): parcel_types[2],
        (1, 1): parcel_types[3]
    }

    # loop through outer 2×2 positions
    for (r, c), ptype in parcel_map.items():
        inner_gs = gridspec.GridSpecFromSubplotSpec(
            2, len(depth_types), subplot_spec=outer_gs[r, c],
            wspace=0.4, hspace=0.6  # <-- more breathing room between rows
        )

        # loop through 2×3 subplots inside each parcel-type block
        for j, depth in enumerate(depth_types):
            # --- TOP ROW: X_List ---
            ax_top = fig.add_subplot(inner_gs[0, j])
            if ptype in results and depth in results[ptype]:
                data_x = results[ptype][depth]["X_List"]
                data_x_mean = np.mean(data_x)
                PlotHistogram(
                    ax_top, data_x,
                    xlabel="X distance (km)",
                    title=f"{ptype} – {depth}\n"
                          + r"$\mu$ = %.2f km" % data_x_mean
                )
                ax_top.axvline((ModelData.xf-ModelData.xf[0])[-1]*1/4,color='green')
                ax_top.set_xlim(left=0, right=(ModelData.xf-ModelData.xf[0])[-1])
                

            # --- BOTTOM ROW: Xdiff_List ---
            ax_bottom = fig.add_subplot(inner_gs[1, j])
            if ptype in results and depth in results[ptype]:
                data_xdiff = results[ptype][depth]["Xdiff_List"]
                data_xdiff_mean = np.mean(data_xdiff)
                PlotHistogram(
                    ax_bottom, data_xdiff,
                    xlabel="X distance from SBF (km)",
                    title=r"$\mu$ = %.2f km" % data_xdiff_mean
                )
                ax_bottom.axvline(0,color='red')
                if ptype != "SBF":
                    halflength = (ModelData.xf-ModelData.xf[0])[-1]/2
                    ax_bottom.set_xlim(left=-halflength, right=halflength)
                elif ptype == "SBF":
                    ax_bottom.set_xlim(left=-10.0, right=10.0)

    # Adjust overall layout to prevent overlap
    fig.subplots_adjust(left=0.06, right=0.97, bottom=0.06, top=0.94)

    AddCrossLines(fig, outer_gs)
    return fig

def PlotAllHistograms_Variables(parcel_types, results):
    """
    Creates a single figure with 4 parcel-type blocks:
      [CL, nonCL]
      [SBF, ColdPool]
    Each block contains a 2×3 grid of subplots (ALL, SHALLOW, DEEP)
    for QV (top) and THv (bottom).
    """

    depth_types = ["ALL", "SHALLOW", "DEEP"]

    # 2×2 outer grid for parcel-type groups
    fig = plt.figure(figsize=(14, 10))
    outer_gs = gridspec.GridSpec(2, 2, figure=fig, wspace=0.2, hspace=0.45)

    # map parcel types into positions
    parcel_map = {
        (0, 0): parcel_types[0],
        (0, 1): parcel_types[1],
        (1, 0): parcel_types[2],
        (1, 1): parcel_types[3]
    }

    # loop through outer 2×2 positions
    for (r, c), ptype in parcel_map.items():
        inner_gs = gridspec.GridSpecFromSubplotSpec(
            2, len(depth_types), subplot_spec=outer_gs[r, c],
            wspace=0.4, hspace=0.55
        )

        # loop through depth types inside each 2×3 block
        for j, depth in enumerate(depth_types):

            # --- TOP ROW: QV (g/kg) ---
            ax_top = fig.add_subplot(inner_gs[0, j])
            if ptype in results and depth in results[ptype]:
                data_qv = np.array(results[ptype][depth]["QV_List"]) * 1e3
                data_qv_mean = np.mean(data_qv)
                PlotHistogram(ax_top, data_qv,
                              xlabel=r"$q_v$ (g/kg)",
                              title=f"{ptype} – {depth}\n"
                              + r"$\mu$ = %.2f g kg$^{-1}$" % data_qv_mean)

            # --- BOTTOM ROW: THv (K) ---
            ax_bottom = fig.add_subplot(inner_gs[1, j])
            if ptype in results and depth in results[ptype]:
                data_th = np.array(results[ptype][depth]["THETA_v_List"])
                data_th_mean = np.mean(data_th)
                PlotHistogram(ax_bottom, data_th,
                              xlabel=r"$\theta_v$ (K)",
                              title=r"$\mu$ = %.2f K" % data_th_mean)
    # Global layout
    fig.subplots_adjust(left=0.06, right=0.97, bottom=0.06, top=0.94)

    AddCrossLines(fig, outer_gs)
    return fig

def AddCrossLines(fig, outer_gs, pad=0.02):
    """
    Draw a shorter cross on the figure.
    pad controls how far the lines stay away from edges (0–0.5).
    """
    # Vertical line: slightly shortened at top & bottom
    fig.lines.append(
        plt.Line2D(
            [0.5, 0.5],            # keep centered
            [pad, 1 - pad],        # shorter vertically
            transform=fig.transFigure,
            color="black", linewidth=1.2
        )
    )

    # Horizontal line: slightly shortened at left & right
    fig.lines.append(
        plt.Line2D(
            [pad, 1 - pad],        # shorter horizontally
            [0.5, 0.5],            # keep centered
            transform=fig.transFigure,
            color="black", linewidth=1.2
        )
    )

In [ ]:
#############################################
#HISTOGRAM PLOTTING
plotting = False #keep false when running job array
# plotting = True

In [ ]:
if plotting:
    pvalues, fig = RunStatisticalComparison(results, test="mannwhitney")
    SaveFigure(
        fig,
        plotType="Project_Algorithms/Tracking_Algorithms/Tracked_Histograms",
        fileName="Tracked_Histograms_StatisticalDifference"
    )

In [ ]:
if plotting:
    parcel_types = ["CL", "nonCL", "SBF","ColdPool"]
    fig = PlotAllHistograms_Heights(parcel_types, results)
    
    axes = fig.get_axes()
    SnapLimitsToTicks(axes, dim='x')
    
    SaveFigure(
        fig,
        plotType="Project_Algorithms/Tracking_Algorithms/Tracked_Histograms",
        fileName="Tracked_Histograms_Heights"
    )

In [ ]:
if plotting:
    parcel_types = ["CL", "nonCL", "SBF", "ColdPool"]
    fig = PlotAllHistograms_Distances(parcel_types, results)
    
    axes = fig.get_axes()
    SetEvenTicks(axes, dim='x', n_ticks=4, decimals=0)
    SnapLimitsToTicks(axes, dim='y')
    
    SaveFigure(
        fig,
        plotType="Project_Algorithms/Tracking_Algorithms/Tracked_Histograms",
        fileName="Tracked_Histograms_Distances"
    )

In [ ]:
if plotting:
    parcel_types = ["CL", "nonCL", "SBF", "ColdPool"]
    fig = PlotAllHistograms_Variables(parcel_types, results)
    
    axes = fig.get_axes()
    MatchAxisLimits(fig.axes[::2], dim='x')
    MatchAxisLimits(fig.axes[1::2], dim='x')
    SnapLimitsToTicks(axes, dim='y')
    
    SaveFigure(
        fig,
        plotType="Project_Algorithms/Tracking_Algorithms/Tracked_Histograms",
        fileName="Tracked_Histograms_Variables"
    )

In [ ]:
#############################################
#JOINT DISTRIBUTIONS
plotting = False #keep false when running job array
plotting = True

In [ ]:
#*testing
# a=results['CL']['SHALLOW'];
# b=results['nonCL']['SHALLOW'];

# fig,axes = plt.subplots(2,2)
# axes[0,0].hist(a['THETA_e_List'])
# axes[0,1].hist(b['THETA_e_List'])
# axes[1,0].hist(a['THETA_e_prime_List'])
# axes[1,1].hist(b['THETA_e_prime_List'])

In [ ]:
def GetParcelData(parcelDepth,parcelType):
    parcelSubset = results[parcelType][parcelDepth]
    X = parcelSubset[f"Xdiff_List"]
    return parcelSubset,X
def GetVariableData(parcelDepth):
    [parcelSubset_CL,X_CL] = GetParcelData(parcelDepth=parcelDepth,parcelType='CL')
    theta_v_CL = parcelSubset_CL['THETA_v_List']
    theta_e_CL = parcelSubset_CL['THETA_e_List']
    [parcelSubset_nonCL,X_nonCL] = GetParcelData(parcelDepth=parcelDepth,parcelType='nonCL')
    theta_v_nonCL = parcelSubset_nonCL['THETA_v_List']
    theta_e_nonCL = parcelSubset_nonCL['THETA_e_List']
    return (theta_v_CL,theta_e_CL, 
     theta_v_nonCL,theta_e_nonCL,
           X_CL,X_nonCL)

def GetVariableData_prime(parcelDepth):
    [parcelSubset_CL,X_CL] = GetParcelData(parcelDepth=parcelDepth,parcelType='CL')
    theta_v_CL = parcelSubset_CL['THETA_v_prime_List']
    theta_e_CL = parcelSubset_CL['THETA_e_prime_List']
    [parcelSubset_nonCL,X_nonCL] = GetParcelData(parcelDepth=parcelDepth,parcelType='nonCL')
    theta_v_nonCL = parcelSubset_nonCL['THETA_v_prime_List']
    theta_e_nonCL = parcelSubset_nonCL['THETA_e_prime_List']
    return (theta_v_CL,theta_e_CL, 
     theta_v_nonCL,theta_e_nonCL,
           X_CL,X_nonCL)

In [ ]:
def GetVariableBinsDictionary(num_bins=500): #*2dHistogram
    num_edges = num_bins + 1
    variableBinsDictionary = {}
    
    configs = { #initial value ranges only
        'THETA_v':     (305, 315, "linear"),
        'THETA_e':     (340, 350, "linear"),
        'THETA_v_prime':     (-6, 6, "linear"),
        'THETA_e_prime':     (-6, 6, "linear"),
        'Xdiff':       (-(512-128)/2, +(512-128)/2, "linear")}
        # 'MSE':         (3.3e5, 3.7e5, "linear"),
        # 'QV':          (0, 20/1e3, "linear")}
        # 'W':           (-20, 50, "signed_log"),
        # 'RH_vapor':    (0, 1.1, "linear"),
        # 'QCQI':        (1e-6, 6e-3, "log"),
        # 'HMC':         (-5e-5, 5e-5, "signed_log"),
        # 'VMF_g':       (-20, 50, "linear"),
        # 'VMF_c':       (-50, 50, "linear")}


    for var, (mn, mx, btype) in configs.items():
        if btype == "linear":
            variableBinsDictionary[var] = np.linspace(mn, mx, num_edges)
            
        elif btype == "log":
            variableBinsDictionary[var] = np.logspace(np.log10(mn), np.log10(mx), num_edges)
            
        elif btype == "signed_log":
            # Determine threshold for the 'linear' center (e.g., 0.1)
            # Ensure we don't start logspace at 0 (log10(0) is undefined)
            threshold = 0.1 if var == 'W' else 1e-4
            
            # Positive side: threshold up to max
            pos = np.logspace(np.log10(threshold), np.log10(mx), num_bins // 2)
            
            # Negative side: min up to -threshold
            # We use abs(mn) to get the log range, then negate and reverse it
            neg = -np.logspace(np.log10(abs(mn)), np.log10(threshold), num_bins // 2)
            
            # neg is currently [-20, ..., -0.1]. These are already increasing.
            # pos is [0.1, ..., 50]. These are already increasing.
            # Combine: [Negatives] + [0] + [Positives]
            variableBinsDictionary[var] = np.concatenate([np.sort(neg), [0], np.sort(pos)])

    return variableBinsDictionary


def Compute2DHistogram(variable1, variable2,
                       variableName1, variableName2,
                       num_bins=100):
    """
    Compute a 2D histogram for two 1D variables using bins from
    GetVariableBinsDictionary().

    variable1 is placed on the x-axis.
    variable2 is placed on the y-axis.
    """

    variableBinsDictionary = GetVariableBinsDictionary(num_bins=num_bins)

    if variableName1 not in variableBinsDictionary:
        raise ValueError(f"{variableName1} not found in variableBinsDictionary")

    if variableName2 not in variableBinsDictionary:
        raise ValueError(f"{variableName2} not found in variableBinsDictionary")

    variable1 = np.asarray(variable1)
    variable2 = np.asarray(variable2)

    validMask = np.isfinite(variable1) & np.isfinite(variable2)

    variable1_valid = variable1[validMask]
    variable2_valid = variable2[validMask]

    variable1_bins = variableBinsDictionary[variableName1]
    variable2_bins = variableBinsDictionary[variableName2]

    histogram2D, variable1_edges, variable2_edges = np.histogram2d(
        variable1_valid,
        variable2_valid,
        bins=[variable1_bins, variable2_bins]
    )

    return histogram2D, variable1_edges, variable2_edges

def Plot2DHistogram(axis,
                    histogram2D, xedges, yedges, 
                    xedges_label, yedges_label,
                    normalize='max',
                    labelSize=10):

    H = histogram2D.copy().astype(float)

    if normalize == "sum":
        totalCount = np.nansum(H)
        if totalCount > 0:
            H = H / totalCount
        else:
            H[:] = np.nan
        cbar_label = "Count / Sum(Count) (%)"

    elif normalize == "max":
        maxValue = np.nanmax(H)
        if maxValue > 0:
            H = H / maxValue
        else:
            H[:] = np.nan
        cbar_label = "Count / Max(Count) (%)"

    elif normalize == "max_x":
        # H from np.histogram2d has shape (nx, ny). 
        # axis=1 finds the max count across all Y-bins for each specific X-bin.
        max_per_column = np.nanmax(H, axis=1, keepdims=True)
        H = np.divide(
            H, max_per_column,
            out=np.full_like(H, np.nan),
            where=max_per_column > 0
        )
        cbar_label = "Count / Max(Count|X) (%)"

    elif normalize == "max_y":
        # H from np.histogram2d has shape (nx, ny). 
        # axis=1 finds the max count across all Y-bins for each specific X-bin.
        max_per_column = np.nanmax(H, axis=0, keepdims=True)
        H = np.divide(
            H, max_per_column,
            out=np.full_like(H, np.nan),
            where=max_per_column > 0
        )
        cbar_label = "Count / Max(Count|Y) (%)"

    elif normalize is None:
        cbar_label = "Count"
    else:
        raise ValueError("normalize must be 'sum', 'max', 'column', or None")

    X, Y = np.meshgrid(xedges, yedges)

    # pcm = axis.pcolormesh(
    #     X, Y, H.T * 100,
    #     cmap='plasma',
    #     shading='auto'
    # )
    vmax_val = np.nanmax(H) if not np.isnan(H).all() else 1.0
    pcm = axis.pcolormesh(
        X, Y, H.T*100,
        norm=LogNorm(vmin=1e-2, vmax=vmax_val*100),
        cmap='plasma',
        shading='auto'
    )

    cbar = plt.colorbar(pcm, ax=axis)
    cbar.set_label(cbar_label, fontsize=labelSize)

    axis.set_xlabel(xedges_label, fontsize=labelSize)
    axis.set_ylabel(yedges_label, fontsize=labelSize)

    return pcm

from matplotlib.colors import LogNorm

In [ ]:
#Plotting Functions

# (1) joint distributions of initial moisture (theta_e) vs buoyancy (theta_v)
def PlotJointDistribution_VariableVariable(perturbation=False):
    GetVariableDataFunction = GetVariableData_prime if perturbation else GetVariableData
    perturbationString="_prime" if perturbation else ""
        
    parcelDepths = ["SHALLOW", "DEEP"]
    parcelTypes = ["CL", "nonCL"]
    
    fig, axes = plt.subplots(
        len(parcelDepths), len(parcelTypes),
        figsize=(12, 9),
        sharex=True,
        sharey=True
    )
    
    for i, parcelDepth in enumerate(parcelDepths):
    
        [theta_v_CL, theta_e_CL, theta_v_nonCL, theta_e_nonCL, _,_] = GetVariableDataFunction(
            parcelDepth=parcelDepth
        )
    
        dataDictionary = {
            "CL":    (theta_v_CL, theta_e_CL),
            "nonCL": (theta_v_nonCL, theta_e_nonCL)
        }
    
        for j, parcelType in enumerate(parcelTypes):
    
            theta_v, theta_e = dataDictionary[parcelType]
    
            hist, theta_v_edges, theta_e_edges = Compute2DHistogram(theta_v, theta_e,
                                                                    f"THETA_v{perturbationString}", f"THETA_e{perturbationString}")
            Plot2DHistogram(
                axes[i, j],
                hist, theta_v_edges, theta_e_edges,
                xedges_label=f"Theta_v{perturbationString} (K)",yedges_label=f"Theta_e{perturbationString} (K)")
    
            axes[i, j].set_title(f"{parcelDepth} {parcelType}")
            if perturbation==True: axes[i,j].axvline(0,color='gray',linestyle='--');axes[i,j].axhline(0,color='gray',linestyle='--')

#(2) joint distribution of initial theta_v/theta_v by distance from SBF and also by distance from CPs.
def PlotJointDistribution_XDistanceVariable(varName="theta_e",
                                            perturbation=False):
    GetVariableDataFunction = GetVariableData_prime if perturbation else GetVariableData
    perturbationString="_prime" if perturbation else ""

    parcelDepths = ["SHALLOW", "DEEP"]
    parcelTypes = ["CL", "nonCL"]

    # --- map variable names ---
    variable_map = {
        f"theta_e{perturbationString}": (f"THETA_e{perturbationString}", f"Theta_e{perturbationString} (K)"),
        f"theta_v{perturbationString}": (f"THETA_v{perturbationString}", f"Theta_v{perturbationString} (K)")
    }

    if varName not in variable_map:
        raise ValueError("varName must be 'theta_e' or 'theta_v'")

    variableKey, variableLabel = variable_map[varName]

    fig, axes = plt.subplots(
        len(parcelDepths), len(parcelTypes),
        figsize=(12, 9),
        sharex=True,
        sharey=True
    )

    for i, parcelDepth in enumerate(parcelDepths):

        theta_v_CL, theta_e_CL, theta_v_nonCL, theta_e_nonCL, X_CL, X_nonCL = GetVariableDataFunction(
            parcelDepth=parcelDepth
        )

        # --- select correct variable ---
        if varName == f"theta_e{perturbationString}":
            dataDictionary = {
                "CL":    (X_CL, theta_e_CL),
                "nonCL": (X_nonCL, theta_e_nonCL)
            }
        elif varName == f"theta_v{perturbationString}":
            dataDictionary = {
                "CL":    (X_CL, theta_v_CL),
                "nonCL": (X_nonCL, theta_v_nonCL)
            }

        for j, parcelType in enumerate(parcelTypes):

            Xdiff, variable = dataDictionary[parcelType]

            hist, x_edges, var_edges = Compute2DHistogram(
                Xdiff, variable,
                "Xdiff", variableKey
            )

            Plot2DHistogram(
                axes[i, j],
                hist, x_edges, var_edges,
                xedges_label="Xdiff (km)",
                yedges_label=variableLabel,
                normalize='max_y'
            )

            axes[i, j].set_title(f"{parcelDepth} {parcelType}")

In [ ]:
if plotting:
    # (1) joint dist1ributions of initial moisture (theta_e) vs buoyancy (theta_v)
    PlotJointDistribution_VariableVariable(perturbation=False)

In [ ]:
if plotting:
    # (1) joint distributions of initial moisture (theta_e') vs buoyancy (theta_v')
    PlotJointDistribution_VariableVariable(perturbation=True)

In [ ]:
if plotting:
    #(2) joint distribution of initial theta_v/theta_v by distance from SBF
    PlotJointDistribution_XDistanceVariable(varName="theta_e",perturbation=False)
    PlotJointDistribution_XDistanceVariable(varName="theta_v",perturbation=False)

In [ ]:
if plotting:
    #(2) joint distribution of initial theta_v/theta_v by distance from SBF
    PlotJointDistribution_XDistanceVariable(varName="theta_e_prime",perturbation=True)
    PlotJointDistribution_XDistanceVariable(varName="theta_v_prime",perturbation=True)

In [ ]:
#############################################
#2D FUNCTIONS
#make a 2d (y,x) binned average-field of q_v/th_v at parcel initial time

In [ ]:
#############################################
#CALCULATING FUNCTIONS

In [ ]:
def SumArray(Dictionary):
    array = np.zeros((ModelData.Nyh,ModelData.Nxh))
    qv_array = array.copy()
    th_v_array = array.copy()
    count = array.copy()
    
    Y_List = Dictionary['Y_List']
    X_List = Dictionary['X_List']
    
    QV_List = Dictionary['QV_List']
    THETA_v_List = Dictionary['THETA_v_List']
    
    array = np.zeros((ModelData.Nyh,ModelData.Nxh))
    count = array.copy()
    
    for (y_kms,x_kms, qv,th_v) in zip(Y_List,X_List, QV_List,THETA_v_List):
        y=np.where(yh==y_kms)
        x=np.where(xh==x_kms)
        
        qv_array[y,x] += qv
        th_v_array[y,x] += th_v
        count[y,x] += 1
    return qv_array, th_v_array, count

def SumArray_SBFLocation(Dictionary):
    x_edges=np.linspace(-256, 256, ModelData.Nxh + 1)

    Ny = ModelData.Nyh
    Nx = len(x_edges) - 1  # number of bins
    
    qv_array = np.zeros((Ny, Nx))
    th_v_array = np.zeros((Ny, Nx))
    count = np.zeros((Ny, Nx))

    Y_List = Dictionary['Y_List']
    Xdiff_List = Dictionary['Xdiff_List']  # <-- SBF-relative distance (km)
    QV_List = Dictionary['QV_List']
    THETA_v_List = Dictionary['THETA_v_List']

    # convert SBF-relative km into bin indices
    x_bin = np.digitize(Xdiff_List, x_edges) - 1
    x_bin = np.clip(x_bin, 0, Nx-1)

    for (y_kms,xidx, qv,th_v) in zip(Y_List, x_bin, QV_List, THETA_v_List):
        y = np.where(yh == y_kms)[0]
        if len(y)==0:
            continue

        qv_array[y, xidx] += qv
        th_v_array[y, xidx] += th_v
        count[y, xidx] += 1

    return qv_array, th_v_array, count
x_edges=np.linspace(-256, 256, ModelData.Nxh + 1)

In [ ]:
def Run2DAverageFields(results):
    outputDictionary = {}

    for outer_key, inner_dict in tqdm(results.items(), desc="Parcel types"):
        outputDictionary[outer_key] = {}
        for inner_key, _ in inner_dict.items():
            Dictionary =  results[outer_key][inner_key]
            
            qv_array, th_v_array, count = SumArray(Dictionary)
            qv_SBFLocation, th_v_SBFLocation, count_SBFLocation = SumArray_SBFLocation(Dictionary)

            outputDictionary[outer_key][inner_key] = {
                "qv_array": qv_array,
                "th_v_array": th_v_array,
                "count": count,

                "qv_SBFLocation": qv_SBFLocation,
                "thv_SBFLocation": th_v_SBFLocation,
                "count_SBFLocation": count_SBFLocation
            }
    return outputDictionary

def OpenOrLoad_2DAverageFields(ModelData, results, overwrite=False):
    fileName = (
        f"2DAverageFields_{ModelData.res}_"
        f"{ModelData.t_res}_{ModelData.Nzh}nz_combined.pkl"
    )
    folderPath = os.path.join(
        DataManager.outputDirectory,
        f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz",
        "2DAverageFields"
    )
    filePath = os.path.join(folderPath, fileName)

    # ---- try to load
    if os.path.exists(filePath) and not overwrite:
        print(f"Loading 2D average fields: {filePath}")
        with open(filePath, "rb") as f:
            outputDictionary = pickle.load(f)
        return outputDictionary

    # ---- otherwise compute
    print("2DAverageFields file not found — computing")
    outputDictionary = Run2DAverageFields(results)

    # ---- ensure directory exists
    os.makedirs(folderPath, exist_ok=True)

    # ---- save output
    with open(filePath, "wb") as f:
        pickle.dump(outputDictionary, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved 2D average fields: {filePath}")
    return outputDictionary

In [ ]:
def TakeAverage(array,count):
    field = np.divide(
        array,
        count,
        out=np.full_like(array, np.nan, dtype=float),
        where=(count != 0)
    )
    return field

In [ ]:
#############################################
#CALCULATING 
running = False
# running = True

In [ ]:
if running:
    outputDictionary = OpenOrLoad_2DAverageFields(ModelData, results)

In [ ]:
##################
#2D PLOTTING FUNCTIONS

In [ ]:
# MakeAxis_Single

plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["figure.titlesize"] = 10

def MakeAxis_Single():
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    return fig, axes

def AddVLines(fig):
    ocean_percent = 1/4
    for ax in fig.get_axes():
        ax.axvline(512*ocean_percent, color='black', linestyle='--', linewidth=1.0)

def PlotSingle_XLocation(parcel_type='CL',parcel_depth='ALL'):
    [fig, axes] = MakeAxis_Single()
    
    qv_array = outputDictionary[parcel_type][parcel_depth]['qv_array']
    th_v_array = outputDictionary[parcel_type][parcel_depth]['th_v_array']
    count = outputDictionary[parcel_type][parcel_depth]['count']
    
    qv_field = TakeAverage(qv_array, count)
    th_v_field = TakeAverage(th_v_array, count)
    
    #QV
    multiplier=1e3
    axis = axes[0,0]
    
    cf = axis.pcolormesh(xh,yh,qv_field*multiplier)
    fig.colorbar(cf, ax=axis)
    axis.set_ylabel('y (km)')
    axis.set_xlabel('x (km)')
    axis.set_xlim(0,ModelData.Nxh);
    axis.set_ylim(0,ModelData.Nyh);
    
    axis = axes[1,0]
    axis.plot(xh,np.nanmean(qv_field,axis=0)*multiplier)
    axis.set_ylabel('qv (g/kg)')
    axis.set_xlabel('x (km)')
    axis.set_xlim(0,ModelData.Nxh);
    
    #TH_v
    multiplier=1
    axis = axes[0,1]
    cf = axis.pcolormesh(xh,yh,th_v_field*multiplier)
    fig.colorbar(cf, ax=axis)
    axis.set_ylabel('y (km)')
    axis.set_xlabel('x (km)')
    axis.set_xlim(0,ModelData.Nxh);
    axis.set_ylim(0,ModelData.Nyh);
    
    axis = axes[1,1]
    axis.plot(xh,np.nanmean(th_v_field,axis=0)*multiplier)
    axis.set_ylabel('th_v (K)')
    axis.set_xlabel('x (km)')
    axis.set_xlim(0,ModelData.Nxh);
    
    AddVLines(fig)

    return fig

def GetBins():
    midpoint = ModelData.xf[-1]/2
    xmin = -midpoint   # km, MUST match SumArray_SBFLocation
    xmax =  midpoint  # km

    N = ModelData.Nxh

    x_edges_SBF = np.linspace(xmin, xmax, N + 1)
    x_centers_SBF = 0.5 * (x_edges_SBF[:-1] + x_edges_SBF[1:])

    return x_edges_SBF, x_centers_SBF

def PlotSingle_SBFLocation(parcel_type='CL', parcel_depth='ALL'):

    [fig, axes] = MakeAxis_Single()

    # load SBF fields
    qv_array = outputDictionary[parcel_type][parcel_depth]['qv_SBFLocation']
    th_v_array = outputDictionary[parcel_type][parcel_depth]['thv_SBFLocation']
    count = outputDictionary[parcel_type][parcel_depth]['count_SBFLocation']

    # take averages safely (handling nan)
    qv_field = TakeAverage(qv_array, count)
    th_v_field = TakeAverage(th_v_array, count)

    # ========= QV =========
    multiplier = 1e3
    ax = axes[0, 0]

    _,x_centers_SBF = GetBins()
    cf = ax.pcolormesh(x_centers_SBF, yh, qv_field * multiplier, shading='auto')
    fig.colorbar(cf, ax=ax)

    ax.set_ylabel("y (km)")
    ax.set_xlabel("SBF-relative x (km)")
    # ax.set_xlim(xmin, xmax)
    ax.set_ylim(0, ModelData.Nyh)

    # line plot
    ax = axes[1, 0]
    ax.plot(x_centers_SBF, np.nanmean(qv_field, axis=0) * multiplier)

    ax.set_ylabel("qv (g/kg)")
    ax.set_xlabel("SBF-relative x (km)")
    # ax.set_xlim(xmin, xmax)

    # ========= THETA_V =========
    ax = axes[0, 1]
    cf = ax.pcolormesh(x_centers_SBF, yh, th_v_field, shading='auto')
    fig.colorbar(cf, ax=ax)

    ax.set_ylabel("y (km)")
    ax.set_xlabel("SBF-relative x (km)")
    # ax.set_xlim(xmin, xmax)
    ax.set_ylim(0, ModelData.Nyh)

    # line plot
    ax = axes[1, 1]
    ax.plot(x_centers_SBF, np.nanmean(th_v_field, axis=0))

    ax.set_ylabel("th_v (K)")
    ax.set_xlabel("SBF-relative x (km)")
    # ax.set_xlim(xmin, xmax)

    # optional: SBF centerline at x=0
    for a in fig.get_axes():
        a.axvline(0, color='k', linestyle='--', linewidth=1)

    fig.tight_layout()
    return fig


In [ ]:
def MakeAxis_All():
    fig, axes = plt.subplots(4, 2, figsize=(10, 8))
    return fig, axes

def PlotAll_XLocation(outputDictionary):
    
    fig, axes = MakeAxis_All()
    
    parcel_order = ["CL", "nonCL", "SBF", "ColdPool"]
    depth_order  = ["ALL", "SHALLOW", "DEEP"]
    
    colors = {
        "ALL":     "black",
        "SHALLOW": "green",
        "DEEP":    "blue"
    }
    
    for row, ptype in enumerate(parcel_order):
        
        # Column 0 → qv
        ax_qv = axes[row, 0]
        
        # Column 1 → th_v
        ax_thv = axes[row, 1]
        
        for depth in depth_order:
            
            Dictionary = outputDictionary[ptype][depth]
            
            qv_array  = Dictionary["qv_array"]
            th_v_array = Dictionary["th_v_array"]
            count = Dictionary["count"]
            
            # TakeAverage returns a 2D field; then take nanmean over y
            qv_field  = TakeAverage(qv_array,count)
            th_v_field = TakeAverage(th_v_array,count)
            
            qv_profile  = np.nanmean(qv_field, axis=0)
            thv_profile = np.nanmean(th_v_field, axis=0)
            
            # plot qv mean profile
            ax_qv.plot(xh, qv_profile * 1e3, 
                       label=depth, color=colors[depth])
            
            # plot th_v mean profile
            ax_thv.plot(xh, thv_profile, 
                        label=depth, color=colors[depth])
            
        # Labels
        ax_qv.set_ylabel(f"{ptype}\nqv (g/kg)")
        ax_thv.set_ylabel(f"{ptype}\nth_v (K)")
        
        ax_qv.set_xlim(0, np.max(xh))
        ax_thv.set_xlim(0, np.max(xh))
        
        ax_qv.grid(alpha=0.3)
        ax_thv.grid(alpha=0.3)
        
        # Legend only on first row
        if row == 0:
            ax_qv.legend(title="Depth", fontsize=8)
            ax_thv.legend(title="Depth", fontsize=8)
    
    # Shared x label
    axes[-1, 0].set_xlabel("x (km)")
    axes[-1, 1].set_xlabel("x (km)")
    
    # Add vertical SBF lines to all axes if needed
    AddVLines(fig)
    
    fig.tight_layout()
    return fig


def PlotAll_SBFLocation(outputDictionary):

    fig, axes = MakeAxis_All()

    parcel_order = ["CL", "nonCL", "SBF", "ColdPool"]
    depth_order  = ["ALL", "SHALLOW", "DEEP"]

    colors = {
        "ALL":     "black",
        "SHALLOW": "green",
        "DEEP":    "blue",
    }

    # get SBF-relative bins
    x_edges_SBF, x_centers_SBF = GetBins()   # MUST return both

    for row, ptype in enumerate(parcel_order):

        ax_qv  = axes[row, 0]   # qv subplot
        ax_thv = axes[row, 1]   # th_v subplot

        for depth in depth_order:

            Dictionary = outputDictionary[ptype][depth]

            qv_array       = Dictionary["qv_SBFLocation"]
            th_v_array     = Dictionary["thv_SBFLocation"]
            count_array    = Dictionary["count_SBFLocation"]

            # 2D averaged fields
            qv_field  = TakeAverage(qv_array, count_array)
            th_v_field = TakeAverage(th_v_array, count_array)

            # 1D profiles (mean over y dim)
            qv_profile  = np.nanmean(qv_field, axis=0)
            thv_profile = np.nanmean(th_v_field, axis=0)

            # ---- plot QV mean profile ----
            ax_qv.plot(
                x_centers_SBF,
                qv_profile * 1e3,
                label=depth,
                color=colors[depth],
                linewidth=2
            )

            # ---- plot THv mean profile ----
            ax_thv.plot(
                x_centers_SBF,
                thv_profile,
                label=depth,
                color=colors[depth],
                linewidth=2
            )

        # ---- axis settings ----
        ax_qv.set_ylabel(f"{ptype}\nqv (g/kg)")
        ax_thv.set_ylabel(f"{ptype}\nth_v (K)")

        ax_qv.grid(alpha=0.3)
        ax_thv.grid(alpha=0.3)

        # Legend only on first row
        if row == 0:
            ax_qv.legend(title="Depth", fontsize=8)
            ax_thv.legend(title="Depth", fontsize=8)

    # shared x labels
    axes[-1, 0].set_xlabel("SBF-relative x (km)")
    axes[-1, 1].set_xlabel("SBF-relative x (km)")

    # add vertical SBF center line at x=0
    for ax in fig.get_axes():
        ax.axvline(0, color='k', linestyle='--', linewidth=1)

    fig.tight_layout()
    return fig

def DeepMinusShallow_SBFLocation(outputDictionary):

    fig, axes = MakeAxis_All()   # 4 rows × 2 columns

    parcel_order = ["CL", "nonCL", "SBF", "ColdPool"]

    # get SBF bins
    x_edges_SBF, x_centers_SBF = GetBins()

    for row, ptype in enumerate(parcel_order):

        ax_qv  = axes[row, 0]
        ax_thv = axes[row, 1]

        # --- extract arrays for SHALLOW and DEEP ---
        dict_sh = outputDictionary[ptype]["SHALLOW"]
        dict_dp = outputDictionary[ptype]["DEEP"]

        # QV fields
        qv_sh = TakeAverage(dict_sh["qv_SBFLocation"], dict_sh["count_SBFLocation"])
        qv_dp = TakeAverage(dict_dp["qv_SBFLocation"], dict_dp["count_SBFLocation"])

        # THv fields
        th_sh = TakeAverage(dict_sh["thv_SBFLocation"], dict_sh["count_SBFLocation"])
        th_dp = TakeAverage(dict_dp["thv_SBFLocation"], dict_dp["count_SBFLocation"])

        # ---- compute mean profiles ----
        qv_sh_prof = np.nanmean(qv_sh, axis=0)
        qv_dp_prof = np.nanmean(qv_dp, axis=0)

        th_sh_prof = np.nanmean(th_sh, axis=0)
        th_dp_prof = np.nanmean(th_dp, axis=0)

        # ---- compute differences ----
        qv_diff = (qv_dp_prof - qv_sh_prof) * 1e3   # g/kg
        thv_diff = th_dp_prof - th_sh_prof          # K

        # ---- plot differences ----
        ax_qv.plot(x_centers_SBF, qv_diff, color="k", linewidth=2)
        ax_thv.plot(x_centers_SBF, thv_diff, color="k", linewidth=2)

        # ---- labels ----
        ax_qv.set_ylabel(f"{ptype}\n(DEEP − SHALLOW)\nqv diff (g/kg)")
        ax_thv.set_ylabel(f"{ptype}\n(DEEP − SHALLOW)\nth_v diff (K)")

        ax_qv.grid(alpha=0.3)
        ax_thv.grid(alpha=0.3)

        # Add zero-line
        ax_qv.axhline(0, color='gray', linestyle='--')
        ax_thv.axhline(0, color='gray', linestyle='--')

        # Vertical SBF line at x=0
        ax_qv.axvline(0, color='k', linestyle='--')
        ax_thv.axvline(0, color='k', linestyle='--')

    # shared x-labels
    axes[-1, 0].set_xlabel("SBF-relative x (km)")
    axes[-1, 1].set_xlabel("SBF-relative x (km)")

    fig.tight_layout()
    return fig


In [ ]:
##################
#2D PLOTTING 
plotting = False #keep false when running job array
# plotting = True

In [ ]:
if plotting:
    fig = PlotAll_XLocation(outputDictionary)

    axes = fig.get_axes()
    SetEvenTicks(axes, dim='y', n_ticks=5, decimals=0)
    SnapLimitsToTicks(axes, dim='y')
    
    SaveFigure(
        fig,
        plotType="Project_Algorithms/Tracking_Algorithms/Tracked_Histograms",
        fileName="Tracked_Histograms_2DAverageFields_XLocation"
    )

In [ ]:
if plotting:
    fig = PlotAll_SBFLocation(outputDictionary)

    axes = fig.get_axes()
    SnapLimitsToTicks(axes, dim='x')
    SetEvenTicks(axes, dim='y', n_ticks=5, decimals=0)
    SnapLimitsToTicks(axes, dim='y')
    
    SaveFigure(
        fig,
        plotType="Project_Algorithms/Tracking_Algorithms/Tracked_Histograms",
        fileName="Tracked_Histograms_2DAverageFields_SBFLocation"
    )

In [ ]:
if plotting:
    fig = DeepMinusShallow_SBFLocation(outputDictionary)

    axes = fig.get_axes()
    SnapLimitsToTicks(axes, dim='x')
    SetEvenTicks(axes, dim='y', n_ticks=5, decimals=0)
    SnapLimitsToTicks(axes, dim='y')
    
    SaveFigure(
        fig,
        plotType="Project_Algorithms/Tracking_Algorithms/Tracked_Histograms",
        fileName="Tracked_Histograms_2DAverageFields_SBFLocation_DEEPminusSHALLOW"
    )

In [ ]:
##################
#TESTING

In [ ]:
# #testing theta_e quantile plot

# def ComputeQuantileData(parcelType1='CL', parcelType2='nonCL',
#                         parcelDepth='DEEP', NumBins=4):

#     # ================================
#     # STEP 0: Load data
#     # ================================
#     variableName = 'THETA_e_List'

#     t_1 = results[parcelType1][parcelDepth]['T_List']
#     t_2 = results[parcelType2][parcelDepth]['T_List']

#     data_1 = results[parcelType1][parcelDepth][variableName]
#     data_2 = results[parcelType2][parcelDepth][variableName]

#     # ================================
#     # STEP 1: Quantile edges (combined)
#     # ================================
#     data_all = np.concatenate([data_1, data_2])
#     QuantileEdges = np.quantile(data_all, np.linspace(0, 1, NumBins + 1))

#     # ================================
#     # STEP 2: Assign bins
#     # ================================
#     CL_bins = np.digitize(data_1, QuantileEdges) - 1
#     nonCL_bins = np.digitize(data_2, QuantileEdges) - 1

#     CL_bins = np.clip(CL_bins, 0, NumBins-1)
#     nonCL_bins = np.clip(nonCL_bins, 0, NumBins-1)

#     # ================================
#     # STEP 3: Count + normalize
#     # ================================
#     CL_counts = np.array([np.sum(CL_bins == b) for b in range(NumBins)])
#     nonCL_counts = np.array([np.sum(nonCL_bins == b) for b in range(NumBins)])

#     CL_fraction = CL_counts / np.sum(CL_counts)
#     nonCL_fraction = nonCL_counts / np.sum(nonCL_counts)

#     return CL_fraction, nonCL_fraction, NumBins


# def PlotQuantileBar(ax, CL_fraction, nonCL_fraction, NumBins, parcelDepth):

#     # ================================
#     # STEP: Plot on axis
#     # ================================
#     x = np.arange(NumBins)
#     width = 0.35

#     ax.bar(x - width/2, CL_fraction, width, label='CL', color='black')
#     ax.bar(x + width/2, nonCL_fraction, width, label='nonCL', color='blue')

#     ax.set_xticks(x)
#     ax.set_xticklabels([f'Q{i+1}' for i in range(NumBins)])
#     ax.set_xlabel('Quantile')
#     ax.set_ylabel('Fraction')
#     ax.set_title(parcelDepth)


# def PlotQuantileBar_Combined():

#     # ================================
#     # CREATE 3x1 SUBPLOT
#     # ================================
#     fig, axes = plt.subplots(3, 1, figsize=(6, 10), sharex=True)

#     depths = ['ALL', 'SHALLOW', 'DEEP']

#     for ax, depth in zip(axes, depths):
#         CL_fraction, nonCL_fraction, NumBins = ComputeQuantileData(
#             'CL', 'nonCL', depth
#         )
#         PlotQuantileBar(ax, CL_fraction, nonCL_fraction, NumBins, depth)

#     axes[0].legend()

#     plt.tight_layout()
#     plt.show()

# def ConvertTimeIndexToHour(timeList):
#     return np.array(timeList)*ModelData.dt/3600+6


# def ComputeQuantileTimeDistribution(parcelType='nonCL', parcelDepth='DEEP',
#                                     NumBins=4, NumTimeBins=20):

#     # ================================
#     # STEP 0: Load data
#     # ================================
#     data_1 = results['CL'][parcelDepth]['THETA_e_List']
#     data_2 = results['nonCL'][parcelDepth]['THETA_e_List']

#     t_1 = results['CL'][parcelDepth]['T_List']
#     t_2 = results['nonCL'][parcelDepth]['T_List']

#     # select based on input
#     if parcelType == 'CL':
#         data = data_1
#         t = t_1
#     elif parcelType == 'nonCL':
#         data = data_2
#         t = t_2
#     else:
#         raise ValueError("parcelType must be 'CL' or 'nonCL'")

#     # ================================
#     # STEP 1: Quantile edges (combined)
#     # ================================
#     data_all = np.concatenate([data_1, data_2])
#     QuantileEdges = np.quantile(data_all, np.linspace(0, 1, NumBins + 1))

#     time_edges = np.linspace(np.min(t), np.max(t), NumTimeBins + 1)

#     # ================================
#     # STEP 2: Assign quantile bins
#     # ================================
#     quantile_bins = np.digitize(data, QuantileEdges) - 1
#     quantile_bins = np.clip(quantile_bins, 0, NumBins-1)

#     # ================================
#     # STEP 3: 2D histogram
#     # ================================
#     H, _, _ = np.histogram2d(
#         quantile_bins,
#         t,
#         bins=[np.arange(NumBins+1), time_edges]
#     )

#     # ================================
#     # STEP 4: Normalize per time
#     # ================================
#     H = H / np.sum(H, axis=0, keepdims=True)

#     return H, time_edges, NumBins


# def PlotQuantileTimeDistribution(ax, H, time_edges, NumBins, parcelType, parcelDepth):

#     # ================================
#     # STEP: Plot on provided axis
#     # ================================
#     cmap = plt.get_cmap('viridis').copy()
#     cmap.set_under('white')

#     im = ax.imshow(H, aspect='auto', origin='lower',
#                    extent=[time_edges[0], time_edges[-1], 0, NumBins],
#                    cmap=cmap,
#                    vmin=1e-6, vmax=1)

#     ax.set_xlabel('Time')
#     ax.set_ylabel('Quantile')
#     ax.set_title(f'{parcelType} {parcelDepth}')

#     yticks = np.arange(NumBins) + 0.5
#     yticklabels = [str(i+1) for i in range(NumBins)]
#     ax.set_yticks(yticks)
#     ax.set_yticklabels(yticklabels)

#     return im

# def PlotQuantileTimeDistribution_Combined():  
#     # ================================
#     # CREATE 2x2 SUBPLOT
#     # ================================
#     fig, axes = plt.subplots(3, 2, figsize=(12, 8),
#                              sharex=True, sharey=True,
#                              constrained_layout=True)
    
#     configs = [
#         ('CL', 'ALL'),
#         ('nonCL', 'ALL'),
#         ('CL', 'SHALLOW'),
#         ('nonCL', 'SHALLOW'),
#         ('CL', 'DEEP'),
#         ('nonCL', 'DEEP')
#     ]
    
#     ims = []
    
#     for ax, (parcelType, parcelDepth) in zip(axes.flat, configs):
#         H, time_edges, NumBins = ComputeQuantileTimeDistribution(parcelType, parcelDepth)
#         im = PlotQuantileTimeDistribution(ax, H, time_edges, NumBins, parcelType, parcelDepth)
#         ims.append(im)
    
#     # ================================
#     # COLORBAR (shared)
#     # ================================
#     cbar = fig.colorbar(ims[0], ax=axes.ravel().tolist(), orientation='vertical',pad=0.01)
#     cbar.set_label('Fraction of parcels')

# PlotQuantileBar_Combined()
# PlotQuantileTimeDistribution_Combined()

In [ ]:
# def PlotCompareCFADs_RowNorm(results, ptype='CL'):
#     qv_min, qv_max = 10/1e3, 20/1e3
#     z_min, z_max = 0, 1 # Your current scale
    
#     qv_bins = np.linspace(qv_min, qv_max, 50)
#     z_bins = np.linspace(z_min, z_max, 50)
    
#     qv_s = np.array(results[ptype]['SHALLOW']['QV_List'])
#     z_s = np.array(results[ptype]['SHALLOW']['Z_List'])
#     qv_d = np.array(results[ptype]['DEEP']['QV_List'])
#     z_d = np.array(results[ptype]['DEEP']['Z_List'])
    
#     # 1. Calculate raw 2D Histograms (counts)
#     H_s, xedges, yedges = np.histogram2d(qv_s, z_s, bins=[qv_bins, z_bins])
#     H_d, _, _ = np.histogram2d(qv_d, z_d, bins=[qv_bins, z_bins])
    
#     # 2. ROW NORMALIZATION (Normalize by height level)
#     # H is [qv_bins, z_bins], so H.T is [z_bins, qv_bins]
#     def row_norm(H):
#         H_new = H.copy()
#         # Sum across qv bins for each z bin
#         row_sums = np.sum(H_new, axis=0) 
#         # Avoid division by zero
#         row_sums[row_sums == 0] = 1
#         return H_new / row_sums

#     H_s_norm = row_norm(H_s)
#     H_d_norm = row_norm(H_d)
    
#     # 3. Plotting
#     fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
#     X, Y = np.meshgrid(xedges, yedges)
    
#     # We plot H.T because pcolormesh expects [Y, X] orientation
#     im0 = axes[0].pcolormesh(X*1e3, Y, H_s_norm.T, cmap='Blues', vmin=0)
#     axes[0].set_title(f'{ptype} SHALLOW (Row-Norm)')
    
#     im1 = axes[1].pcolormesh(X*1e3, Y, H_d_norm.T, cmap='Blues', vmin=0)
#     axes[1].set_title(f'{ptype} DEEP (Row-Norm)')
    
#     # Difference of probabilities
#     diff = H_d_norm - H_s_norm
#     limit = np.nanmax(np.abs(diff))
#     im2 = axes[2].pcolormesh(X*1e3, Y, diff.T, cmap='RdBu_r', vmin=-limit, vmax=limit)
#     axes[2].set_title('Diff (Deep - Shallow)')

#     # Formatting
#     plt.colorbar(im0, ax=axes[0], label='P(qv | Z)')
#     plt.colorbar(im1, ax=axes[1], label='P(qv | Z)')
#     plt.colorbar(im2, ax=axes[2], label='$\Delta$ Probability')
    
#     for ax in axes:
#         ax.set_xlabel('$q_v$ (g/kg)')
#     axes[0].set_ylabel('Height index / Z')
    
#     return fig

# PlotCompareCFADs_RowNorm(results)